# Milestone 1: Formula 1 Podium Prediction Companion
**Course:** CSEN 903 · Advanced Machine Learning & Data Engineering  
**Project:** F1 World Championship (1950–2024) Podium Prediction Pipeline  
**Target:** Binary classification — whether a driver finishes on the podium (`positionOrder <= 3`).  
**Prediction Temporal Cutoff:** Strictly *after qualifying, before race start* (lights out). Current-race outcome fields and post-race standings are strictly forbidden as features.

---

### Team Ownership Matrix

| Member | Role | Pipeline Ownership |
| :--- | :--- | :--- |
| **M1** | Data Lead | Raw table audit, referential integrity, cleaning, duplicate resolution, before/after EDA, report assembly |
| **M2** | Analytics Lead | Data-engineering multi-hop questions (Q1–Q3), domain mappings, Explainable AI (XAI) suite |
| **M3** | Features Lead | Leakage-safe feature engineering, temporal validation, preprocessing pipelines, order experiments, ablation |
| **M4** | Modeling Lead | Heuristic & statistical baselines, Shallow Neural Network, hyperparameter tuning, threshold calibration, inference function |

---

### Pipeline Roadmap
1. **Shared Components & Global Configuration** *(Current Section)*
2. **Data Audit & Integrity Verification** *(Step 1 · M1)*
3. **Data Cleaning & Grain Resolution** *(Step 2 · M1)*
4. **Before vs. After Exploratory Data Analysis (EDA)** *(Step 2 · M1)*
5. **Data-Engineering Analytical Questions (Q1–Q3)** *(Step 3 · M2)*
6. **Leakage-Safe Feature Engineering** *(Step 4 · M3)*
7. **Preprocessing Pipelines & Splitting Experiments** *(Step 5 · M3)*
8. **Predictive Modeling Benchmarks & Tuning** *(Step 6 · M4)*
9. **Feature Group Ablation Study** *(Step 7 · M3)*
10. **Explainable AI (XAI: SHAP & LIME)** *(Step 8 · M2)*
11. **Operational Inference Interface** *(Step 9 · M4)*
12. **Limitations & Project Summary** *(Step 10 · All)*

## 1. Shared Components (Day 1 Agreement)

To maintain strict pipeline integrity across all team members and prevent data leakage or environment mismatches, all core configurations, data contracts, and visual standards are formalized below:

1. **Config Cell**: Single source of truth for random seeds, split cutoffs, file paths, and historical race scope flags.
2. **Leakage Blacklist**: An enforced registry of current-race outcome fields across all 14 tables that must never enter feature engineering.
3. **Modeling Data Contract**: Strict specification of the modeling grain `(raceId, driverId)`, target column, identifier lineage, and `FEATURE_GROUPS` dictionary.
4. **Unified Visual Style & Plot Exporter**: Publication-quality Seaborn/Matplotlib styling exporting 300-DPI figures directly to `figures/`.
5. **Decision Log**: Traceable registry of all architectural, scope, and algorithmic choices.

In [ ]:
# ==============================================================================
# 1.1 SHARED CONFIGURATION & ENVIRONMENT SETUP
# ==============================================================================
from dataclasses import dataclass
from pathlib import Path
import os
import sys
import re
import warnings
from typing import Dict, List, Set, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Suppress minor warnings for clean presentation
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Set global random seed for complete reproducibility
RANDOM_SEED: int = 42
np.random.seed(RANDOM_SEED)

# Temporal Split Boundaries (Temporal splitting prevents regulation leakage)
# Train: <= 2019 | Validation: 2020-2021 | Test: >= 2022 (Modern ground-effect era)
TRAIN_MAX_YEAR: int = 2019
VAL_YEARS: Tuple[int, int] = (2020, 2021)
TEST_MIN_YEAR: int = 2022

# Scope & Cleaning Flags
EXCLUDE_INDY_500: bool = True  # Exclude 1950-1960 Indy 500 rounds (non-F1 formula)
SENTINEL_VALUES: List[str] = [r"\N", "\\N", ""]

# Output directories
FIGURES_DIR: Path = Path("figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Dynamic Dataset Path Resolution (Kaggle & Local fallback)
def resolve_dataset_dir() -> Path:
    candidates = [
        Path("/kaggle/input/datasets/rohanrao/formula-1-world-championship-1950-2020"),
        Path("/kaggle/input/formula-1-world-championship-1950-2020"),
        Path("./archive"),
        Path("../archive"),
        Path("d:/GUC/Semester 9/ACL/Project/Formula 1/archive"),
    ]
    for p in candidates:
        if p.exists() and (p / "results.csv").exists():
            return p.resolve()
    raise FileNotFoundError(
        f"F1 dataset directory could not be located. Checked paths:\n"
        + "\n".join(f" - {p}" for p in candidates)
    )

DATASET_DIR: Path = resolve_dataset_dir()

# Set Pandas presentation options
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 1000)

print(f"[OK] Random Seed        : {RANDOM_SEED}")
print(f"[OK] Dataset Directory  : {DATASET_DIR}")
print(f"[OK] Figures Directory  : {FIGURES_DIR.resolve()}")
print(f"[OK] Split Years        : Train <= {TRAIN_MAX_YEAR} | Val {VAL_YEARS} | Test >= {TEST_MIN_YEAR}")
print(f"[OK] Exclude Indy 500   : {EXCLUDE_INDY_500}")

In [ ]:
# ==============================================================================
# 1.2 UNIFIED VISUALIZATION STYLING & FIGURE EXPORT HELPER
# ==============================================================================

# Custom F1-inspired, publication-grade visual theme
F1_PALETTE = {
    "crimson": "#E10600",      # Formula 1 Official Red
    "dark": "#15151E",         # Charcoal Black
    "teal": "#00D2BE",         # Mercedes / Modern Cyan
    "blue": "#1E41FF",         # Red Bull / Alpine Blue
    "orange": "#FF8700",       # McLaren Papaya
    "green": "#229971",        # Aston Martin Green
    "gray": "#7A7A85",         # Neutral Muted Gray
    "light_gray": "#E0E0E6",   # Background Grid Line Gray
}

# Apply global Matplotlib & Seaborn styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 100,
    "figure.autolayout": False,
    "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Helvetica", "Arial"],
    "font.family": "sans-serif",
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.titlepad": 14,
    "axes.labelsize": 12,
    "axes.labelweight": "medium",
    "axes.labelpad": 8,
    "axes.edgecolor": "#CCCCCC",
    "axes.linewidth": 1.0,
    "grid.color": "#E5E5EB",
    "grid.linestyle": "--",
    "grid.alpha": 0.7,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.frameon": True,
    "legend.framealpha": 0.9,
    "legend.edgecolor": "#E5E5EB",
})

def save_fig(
    fig_id: str,
    tight_layout: bool = True,
    fig_extension: str = "png",
    dpi: int = 300,
    transparent: bool = False
) -> Path:
    """
    Saves the active matplotlib figure to the shared FIGURES_DIR with consistent formatting.
    
    Parameters
    ----------
    fig_id : str
        Unique snake_case identifier for the figure (e.g., 'q1_front_row_conversion').
    tight_layout : bool, default=True
        Whether to invoke plt.tight_layout() prior to export.
    fig_extension : str, default='png'
        File extension format ('png', 'pdf', 'svg').
    dpi : int, default=300
        Resolution for publication-quality output.
    transparent : bool, default=False
        Whether background should be saved as transparent.
        
    Returns
    -------
    Path : The saved file path.
    """
    file_path = FIGURES_DIR / f"{fig_id}.{fig_extension}"
    if tight_layout:
        plt.tight_layout()
    plt.savefig(file_path, format=fig_extension, dpi=dpi, transparent=transparent, bbox_inches="tight")
    print(f"[FIGURE SAVED] {file_path}")
    return file_path

print("[OK] Publication plot theme and save_fig() helper initialized.")

In [ ]:
# ==============================================================================
# 1.3 DATA LEAKAGE BLACKLIST & INTEGRITY GUARD (AUDITED ACROSS ALL 14 TABLES)
# ==============================================================================

# Core Blacklist: Columns strictly generated during or after the race.
# These can NEVER be used as features for predicting pre-race podium probability.
LEAKAGE_BLACKLIST_COLUMNS: Set[str] = {
    # 1. Current-race direct outcomes from results.csv
    "position",         # Race finishing position (e.g. '1', '2', '\N')
    "positionText",     # Finishing status text ('R', 'D', 'W', etc.)
    "positionOrder",    # Finishing order integer -> TARGET ONLY (podium = positionOrder <= 3)
    "points",           # Current race points scored (raw column name)
    "laps",             # Completed race laps (raw column name)
    "time",             # Race finish time / gap string (raw column name)
    "milliseconds",     # Total race time in milliseconds (raw column name)
    "fastestLap",       # Lap number on which fastest lap was set
    "rank",             # Fastest lap rank among drivers in the race
    "fastestLapTime",   # Lap time string of fastest lap in race
    "fastestLapSpeed",  # Average speed of fastest lap in race
    "statusId",         # Current race finishing/retirement status ID
    
    # 2. Joined outcome descriptions from status.csv & constructor_results.csv
    "status",           # Outcome description ('Finished', 'Engine', 'Collision', etc.)
    
    # 3. In-race telemetry from lap_times.csv & pit_stops.csv
    "lap",              # Lap number during race
    "stop",             # Pit stop number
    "duration",         # Pit stop duration
    
    # 4. Standings collisions (unlagged post-race championship standings)
    "wins",             # Post-race cumulative wins
    "post_race_points",
    "post_race_position",
    "post_race_wins",
    "post_race_rank",
}

# Regex patterns for catching accidental post-race or in-race column variants
LEAKAGE_PATTERNS: List[re.Pattern] = [
    re.compile(r"^fastest.*", re.IGNORECASE),
    re.compile(r"^post_.*", re.IGNORECASE),
    re.compile(r".*current.*", re.IGNORECASE),
    re.compile(r"^finish.*", re.IGNORECASE),
    re.compile(r"^pit_stop_.*", re.IGNORECASE),
    re.compile(r"^lap_time_.*", re.IGNORECASE),
]

# Table-by-table schema audit reference mapping:
TABLE_TEMPORAL_AUDIT: Dict[str, str] = {
    "circuits": "PRE-RACE SAFE: Static track characteristics (country, location, altitude, lat, lng).",
    "constructors": "PRE-RACE SAFE: Static constructor identity (name, nationality).",
    "drivers": "PRE-RACE SAFE: Static driver identity (dob/age, nationality).",
    "seasons": "PRE-RACE SAFE: Season year metadata.",
    "races": "PRE-RACE SAFE: Scheduled race calendar, round, circuit, session timestamps.",
    "qualifying": "PRE-RACE SAFE: Concluded before lights-out. Column 'position' MUST be renamed to 'quali_position'.",
    "results": "HYBRID: 'grid', 'constructorId', 'number' are SAFE; 'positionOrder' is TARGET; 11 outcome columns are BLACKLISTED.",
    "status": "POST-RACE LEAKAGE: Describes current race termination reason. Safe only for historical aggregations.",
    "constructor_results": "POST-RACE LEAKAGE: Records current race constructor points. Safe only lagged to t-1.",
    "constructor_standings": "POST-RACE LEAKAGE: Ergast records post-race standings for raceId. Safe only lagged to round t-1.",
    "driver_standings": "POST-RACE LEAKAGE: Ergast records post-race standings for raceId. Safe only lagged to round t-1.",
    "lap_times": "IN-RACE TELEMETRY: 100% Forbidden for current race.",
    "pit_stops": "IN-RACE EVENTS: 100% Forbidden for current race.",
    "sprint_results": "PRE-RACE SAFE (CONDITIONAL): Saturday sprint before Sunday race; features must be prefixed with 'sprint_'.",
}

def assert_leakage_safe(feature_cols: List[str]) -> None:
    """
    Validates candidate feature columns against the comprehensive leakage blacklist.
    Raises ValueError immediately if any blacklisted column or pattern is detected.
    """
    violations = []
    for col in feature_cols:
        col_clean = col.strip()
        if col_clean in LEAKAGE_BLACKLIST_COLUMNS:
            violations.append(f"Direct Blacklist Match: '{col_clean}'")
        for pattern in LEAKAGE_PATTERNS:
            if pattern.match(col_clean):
                violations.append(f"Pattern Match [{pattern.pattern}]: '{col_clean}'")
                
    if violations:
        error_msg = "\n".join(f"  ❌ {v}" for v in violations)
        raise ValueError(
            f"[CRITICAL DATA LEAKAGE DETECTED]\n"
            f"The following feature columns violate pre-race temporal cutoff:\n{error_msg}"
        )
    print(f"[LEAKAGE CHECK PASSED] All {len(feature_cols)} candidate features strictly respect pre-race boundary.")

# Sanity tests: Ensure validator catches direct, pattern, and standings collision leaks
test_failures = []
for forbidden in ["milliseconds", "position", "status", "fastestLapTime", "wins"]:
    try:
        assert_leakage_safe(["grid", "quali_position", forbidden])
        test_failures.append(forbidden)
    except ValueError:
        pass

if not test_failures:
    print("[TEST CONFIRMED] Leakage guard successfully verified against test forbidden columns.")
else:
    raise AssertionError(f"Leakage guard failed to catch: {test_failures}")

In [ ]:
# ==============================================================================
# 1.4 MODELING DATA CONTRACT & FEATURE GROUPS REGISTRY
# ==============================================================================

# 1. Grain & Lineage Identifiers (retained for indexing, never used as numeric features)
IDENTIFIER_COLUMNS: List[str] = [
    "raceId",
    "driverId",
    "constructorId",
    "date",
    "year",
]

# 2. Prediction Target
TARGET_COLUMN: str = "podium"  # Binary: 1 if positionOrder <= 3 else 0

# 3. Canonical Feature Groups Mapping
# Used for feature engineering, ablation studies, and XAI grouping.
FEATURE_GROUPS: Dict[str, List[str]] = {
    "grid": [
        "grid",
        "pit_start",
        "front_row",
        "teammate_grid_diff",
    ],
    "qualifying": [
        "quali_position",
        "gap_to_pole",
        "q3_reached",
        "missing_quali_flag",
    ],
    "driver_form": [
        "driver_rolling_podium_rate_5",
        "driver_rolling_avg_finish_5",
        "driver_career_podium_rate",
    ],
    "constructor_form": [
        "constructor_rolling_podium_rate_5",
        "constructor_rolling_points_5",
    ],
    "standings": [
        "driver_pre_race_points",
        "driver_pre_race_rank",
        "constructor_pre_race_points",
        "constructor_pre_race_rank",
    ],
    "context": [
        "field_size",
        "round",
        "is_home_race",
        "circuit_historical_podium_rate",
    ],
    "sprint": [
        "sprint_position",
        "sprint_points",
    ],
}

def validate_data_contract(
    df: pd.DataFrame,
    feature_cols: Optional[List[str]] = None,
    enforce_grain_only: bool = False
    ) -> bool:
    """
    Enforces the modeling table data contract:
    1. Composite grain uniqueness on (raceId, driverId).
    2. Presence and non-nullity of identifier columns.
    3. Target column presence, non-nullity, and binary values {0, 1}.
    4. Features pass the leakage guard.
    """
    # Check grain
    if not {"raceId", "driverId"}.issubset(df.columns):
        raise AssertionError("Dataframe is missing composite key columns 'raceId' or 'driverId'.")
    
    dup_count = df.duplicated(subset=["raceId", "driverId"]).sum()
    if dup_count > 0:
        raise AssertionError(
            f"Grain violation: Found {dup_count} duplicate (raceId, driverId) rows! "
            f"Analytical table must have strictly one row per driver per race."
        )
        
    if enforce_grain_only:
        print(f"[CONTRACT PASSED] Grain strictly verified: {len(df):,} unique (raceId, driverId) entries.")
        return True
        
    # Check identifiers
    for col in IDENTIFIER_COLUMNS:
        if col not in df.columns:
            raise AssertionError(f"Missing required identifier column: '{col}'")
        if df[col].isnull().any():
            raise AssertionError(f"Identifier column '{col}' contains unexpected NULL values.")
            
    # Check target
    if TARGET_COLUMN not in df.columns:
        raise AssertionError(f"Target column '{TARGET_COLUMN}' is missing from modeling table.")
    if df[TARGET_COLUMN].isnull().any():
        raise AssertionError(f"Target column '{TARGET_COLUMN}' contains NULL values.")
    unique_targets = set(df[TARGET_COLUMN].dropna().unique())
    if not unique_targets.issubset({0, 1}):
        raise AssertionError(f"Target '{TARGET_COLUMN}' must be strictly binary {{0, 1}}, found: {unique_targets}")
        
    # Validate feature columns if provided
    if feature_cols is not None:
        assert_leakage_safe(feature_cols)
        
    print(f"[CONTRACT PASSED] Full modeling table contract satisfied across {len(df):,} rows.")
    return True

print("[OK] Data contract rules and FEATURE_GROUPS registry established.")

### 1.5 Shared Decision Log & 14-Table Leakage Reference

The table below embeds the shared architectural and pipeline decisions established for Milestone 1. The persistent version is tracked in [`docs/decision_log.md`](docs/decision_log.md).

| Decision ID | Pipeline Stage | Owner | Decision | Rationale & Evidence | Status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **DEC-001** | Shared Setup | All | Global Random Seed = `42` | Ensures deterministic reproduction across all environments and runs. | **Approved** |
| **DEC-002** | Shared Setup | M3 / All | Temporal Split: Train $\le 2019$, Val $2020\text{–}2021$, Test $\ge 2022$ | Prevents temporal leakage across F1 aero/engine regulation eras; 2022 tests ground-effect generalization. | **Approved** |
| **DEC-003** | Scope / Cleaning | M1 | Exclude Indianapolis 500 ($1950\text{–}1960$) | Run under AAA/USAC Indy regulations on an oval; non-F1 grid contaminates conversion & form baselines. | **Approved** |
| **DEC-004** | Leakage Guard | M3 / M4 | Strict pre-race temporal cutoff & Blacklist | Prediction target is `positionOrder <= 3`. All current-race in-event metrics across 14 tables are forbidden. | **Approved** |
| **DEC-005** | Data Contract | All | Table Grain strictly `(raceId, driverId)` | Exactly one prediction per driver-race entry. Identifiers retained for lineage, target is binary `podium`. | **Approved** |
| **DEC-006** | Environment | M1 | Kaggle & Local path auto-detection | Seamless execution across Kaggle notebooks and local git repositories with zero config tweaks. | **Approved** |
| **DEC-007** | Visual Standards | M2 | Publication-grade Seaborn style & `save_fig()` | Enforces visual cohesion and automatic 300-DPI exports to `figures/` for report assembly. | **Approved** |

#### 14-Table Leakage Audit Summary
- **Safe Pre-Race Tables:** `circuits`, `constructors`, `drivers`, `seasons`, `races`, `qualifying` (must rename `position` to `quali_position`).
- **Target & In-Race Table (`results`):** Only `grid`, `constructorId`, `number` are pre-race features; `positionOrder` defines target; remaining 11 columns are strictly blacklisted.
- **In-Race Telemetry Tables (100% Forbidden):** `lap_times`, `pit_stops`.
- **Post-Race Ergast Standings Tables:** `driver_standings`, `constructor_standings`, `constructor_results` are updated *after* `raceId`. Must be lagged to round $t-1$.

In [ ]:
# ==============================================================================
# 1.6 RAW DATASET LOADER (CLEAN COPY PATTERN)
# ==============================================================================

def load_raw_datasets(dataset_dir: Path, sentinels: List[str] = SENTINEL_VALUES) -> Dict[str, pd.DataFrame]:
    """
    Loads all 14 relational CSV tables with standardized sentinel values into memory.
    
    CRITICAL RULE:
    The returned dataframes in raw_dataframes must remain 100% IMMUTABLE.
    All downstream auditing, cleaning, and transformations must operate strictly 
    on explicit copies (.copy()) to preserve original raw state for before/after EDA.
    """
    csv_paths = sorted(dataset_dir.glob("*.csv"))
    if not csv_paths:
        raise FileNotFoundError(f"No CSV files found in {dataset_dir}")
        
    tables: Dict[str, pd.DataFrame] = {}
    summary_rows = []
    
    for file_path in csv_paths:
        table_name = file_path.stem
        # Read with sentinel handling
        df = pd.read_csv(file_path, na_values=sentinels, low_memory=False)
        tables[table_name] = df
        
        summary_rows.append({
            "Table Name": table_name,
            "File Name": file_path.name,
            "Rows": f"{len(df):,}",
            "Columns": len(df.columns),
            "Memory (MB)": f"{df.memory_usage(deep=True).sum() / (1024 * 1024):.2f}",
        })
        
    summary_df = pd.DataFrame(summary_rows)
    print(f"[SUCCESS] Loaded {len(tables)} relational F1 tables into memory.")
    display(summary_df)
    return tables

# Execute the raw loader
raw_dataframes = load_raw_datasets(DATASET_DIR)

---

## Next Steps: Transition to Step 1 (Data Audit & Inspection)

With the **Shared Components** agreed upon and initialized:
* **Global config & random seeds** are fixed.
* **Leakage blacklist & contract guards** are active.
* **Visual styling & figure exporter** are configured.
* **Raw immutable dataframes** are loaded into `raw_dataframes`.

The pipeline advances to **Step 1: Data Audit & Initial Inspection** (led by M1):
- Comprehensive missingness analysis and sentinels audit.
- Primary key uniqueness & referential foreign-key anti-join checks across all 14 tables.
- Pre-cleaning baseline EDA (season counts, entrant volume, qualifying coverage drift, and historical podium base rates).